# Best On-Device SLM for GridRoute + MazeBench -- Kaggle Training Notebook

Runs the full pipeline from `neuro-symbolic-pathfinding` end to end on a free Kaggle GPU: Phase 1 baselines (Gemma 4 E2B/E4B + AlphaMaze on GridRoute 5x5 and MazeBench), then Phase 2 training recipes on Gemma 4 E2B. See `idea.md`, `refined-idea.md`, and `experiment-plan.md` in the repo for the full plan this executes. No specific technique is the point here -- get Gemma 4 as good as possible at both benchmarks, report whichever recipe actually works.

**Before running:**

1. Settings (right panel) -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> **Internet: On** (needed for pip installs, HuggingFace downloads, and cloning the repo).
3. Give this notebook access to the repo (it's private). Two options:
   - **Recommended**: Add-ons -> Secrets -> add a secret named `GITHUB_TOKEN` with a GitHub [fine-grained personal access token](https://github.com/settings/tokens) scoped read-only to this one repo. Never paste the token directly into a cell -- Kaggle Secrets keeps it out of the notebook file entirely, and this notebook only ever reads it through `kaggle_secrets`.
   - **Alternative**: zip the repo yourself, upload it as a private Kaggle Dataset, attach it to this notebook (Add data -> your dataset), and change `REPO_DIR` in the next cell to wherever Kaggle mounts it (typically `/kaggle/input/<dataset-name>`), then skip the clone cell.
4. Kaggle's free tier: about 30 GPU-hours/week, 9-12h max per session. Gemma 4 **E4B** may not fit the T4 at all (its LoRA footprint has been reported elsewhere as ~17GB, over the T4's 16GB) -- the feasibility check cell below confirms this on the real hardware before you commit to it. The **consistency** GRPO condition roughly doubles per-step cost -- run the timing test cell before committing to a full run.

This notebook is a thin runner around the actual project code (`eval.py`, `train_sft.py`, `train_grpo.py`) -- it doesn't reimplement any of the logic, so if you change the training/eval logic, edit those files and re-clone/re-sync rather than editing this notebook's cells directly.

In [ ]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        clone_url = f"https://{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


In [ ]:
%pip install -q -r requirements.txt
%pip install -q unsloth


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## Feasibility check: does Gemma 4 E2B/E4B LoRA actually fit this GPU?

Don't assume the ~8-10GB (E2B) / ~17GB (E4B) numbers from elsewhere -- confirm on this exact hardware.

In [ ]:
!python check_finetune_feasibility.py


## Phase 1: baselines -- Gemma 4 E2B/E4B + AlphaMaze on MazeBench and GridRoute 5x5

AlphaMaze should land near its published 93% on MazeBench (using their real scoring code via the submodule) -- this is the harness sanity check. Gemma 4's numbers on both benchmarks are the actual open question: no SLM has been tested on GridRoute anywhere in the literature found so far.

In [ ]:
import os
os.makedirs("data/models", exist_ok=True)
if not os.path.isdir("data/models/alphamaze-v0.2-1.5b"):
    from huggingface_hub import snapshot_download
    snapshot_download("homebrewltd/AlphaMaze-v0.2-1.5B", local_dir="data/models/alphamaze-v0.2-1.5b")
print("AlphaMaze checkpoint ready.")


In [ ]:
!python eval.py --model alphamaze --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model alphamaze --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


In [ ]:
!python eval.py --model gemma4-e2b --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model gemma4-e2b --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


E4B baseline (inference-only, no LoRA) is much lighter than E4B *training* -- run this regardless of what the feasibility check said about training, since it doesn't need Unsloth or LoRA attachment, just enough VRAM to hold the weights at 4-bit.

In [ ]:
!python eval.py --model gemma4-e4b --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model gemma4-e4b --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, then a short GRPO timing test before committing to the full run -- see `experiment-plan.md`'s Immediate Next Step.

In [ ]:
!python train_sft.py --model gemma4-e2b --format nl --grid_size 5 --n_tasks 400 --epochs 1 --output_dir ./results/sft_gemma4-e2b_single


In [ ]:
!python train_grpo.py --model gemma4-e2b --condition single --grid_size 5 --n_tasks 20 --max_steps 20 --output_dir ./results/grpo_gemma4-e2b_single_timing


Check the timing printout above (extrapolated GPU-hours for 1000 steps) before running the full step count below. Adjust `--max_steps` in the next cell if the full run won't fit your remaining quota.

In [ ]:
!python train_grpo.py --model gemma4-e2b --condition single --grid_size 5 --n_tasks 50 --max_steps 200 --output_dir ./results/grpo_gemma4-e2b_single


In [ ]:
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_single --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_single --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [ ]:
!python train_sft.py --model gemma4-e2b --format mixed --grid_size 5 --n_tasks 400 --epochs 1 --output_dir ./results/sft_gemma4-e2b_mixed


In [ ]:
!python train_grpo.py --model gemma4-e2b --condition mixed --grid_size 5 --n_tasks 50 --max_steps 200 --output_dir ./results/grpo_gemma4-e2b_mixed


In [ ]:
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_mixed --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_mixed --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism.

In [ ]:
!python train_grpo.py --model gemma4-e2b --condition consistency --grid_size 5 --n_tasks 20 --max_steps 20 --output_dir ./results/grpo_gemma4-e2b_consistency_timing


This condition generates an extra partner completion per reward call, so it costs roughly double the single/mixed conditions per step. Check the timing printout above before committing to a full run -- if your remaining weekly GPU quota is tight, this is the recipe to scale down or skip first (it's one candidate among three, not the point of the project).

In [ ]:
!python train_grpo.py --model gemma4-e2b --condition consistency --grid_size 5 --n_tasks 50 --max_steps 200 --output_dir ./results/grpo_gemma4-e2b_consistency


In [ ]:
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_consistency --benchmark mazebench --n 100 --output_dir ./results/eval
!python eval.py --model gemma4-e2b --checkpoint ./results/grpo_gemma4-e2b_consistency --benchmark gridroute-nl --grid_size 5 --n 50 --output_dir ./results/eval


## Aggregate results into one comparison table

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("./results/eval/*.json")):
    with open(path) as f:
        d = json.load(f)
    label = path.split("/")[-1]
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n")}
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("./results/comparison_table.csv", index=False)
df


## Download results

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "./results")
print("Saved: /kaggle/working/results.zip -- download it from the Kaggle output panel on the right.")
